# 🕐 **Earliest Deadline First (EDF):**  
Exploring how does it perform when the priority queue has 32 tasks.

In [25]:
import heapq
import time
import numpy as np
from typing import List, Dict
from dataclasses import dataclass
from collections import defaultdict
import random

# --- Data Structures ---

@dataclass
class TaskConfig:
    task_id: int
    arrival_time: float
    expected_duration: float
    deadline: float
    is_io_bound: bool = False
    is_foreground: bool = False  # OS decides boost based on this


@dataclass
class TaskState:
    base_priority: int = 0
    dynamic_priority: int = 0
    start_time: float = -1
    end_time: float = -1
    remaining: float = 0
    executed_time: float = 0
    waiting_time: float = 0

    def __post_init__(self):
        self.dynamic_priority = self.base_priority


class Task:
    def __init__(self, config: TaskConfig):
        self.config = config
        self.state = TaskState(remaining=config.expected_duration)
        self.assign_base_priority()

    def assign_base_priority(self):
        """
        Simulate OS assigning priority based on characteristics
        """
        priority = 8  # default: NORMAL
        if self.config.deadline < 10:
            priority += 4  # tight deadline → higher priority
        if self.config.is_io_bound:
            priority += 2
        if self.config.is_foreground:
            priority += 2
        self.state.base_priority = min(priority, 15)
        self.state.dynamic_priority = self.state.base_priority

    def __lt__(self, other):
        return self.state.dynamic_priority > other.state.dynamic_priority


# --- Metrics ---

class SchedulerMetrics:
    def __init__(self):
        self.task_metrics: Dict[int, Dict] = defaultdict(dict)
        self.system_metrics = {
            "total_context_switches": 0,
            "cpu_utilization": 0,
            "missed_deadlines": 0,
            "jains_fairness": 0,
            "avg_waiting_time": 0,
            "avg_response_time": 0
        }

    def calculate_jains_fairness(self, execution_times: List[float]):
        numerator = sum(execution_times) ** 2
        denominator = len(execution_times) * sum(t ** 2 for t in execution_times)
        return numerator / denominator if denominator != 0 else 0


# --- Scheduler ---

QUANTUM_BASE = 2

class Scheduler:
    def __init__(self):
        self.clock = 0
        self.ready_queue = []
        self.current_task = None
        self.metrics = SchedulerMetrics()
        self.pending_tasks = []
        self.total_busy_time = 0

    def add_task(self, task: Task):
        heapq.heappush(self.pending_tasks, (task.config.arrival_time, task))

    def _update_priorities(self):
        if self.current_task:
            if self.current_task.config.is_io_bound and random.random() < 0.6:
                self.current_task.state.dynamic_priority = min(
                    self.current_task.state.dynamic_priority + 2, 15)
            elif self.clock % 4 == 0:
                self.current_task.state.dynamic_priority = max(
                    self.current_task.state.base_priority,
                    self.current_task.state.dynamic_priority - 1
                )

    def _handle_preemption(self):
        if self.current_task and self.ready_queue:
            next_task = self.ready_queue[0]
            if next_task.state.dynamic_priority > self.current_task.state.dynamic_priority:
                heapq.heappush(self.ready_queue, self.current_task)
                self.current_task = heapq.heappop(self.ready_queue)
                self.metrics.system_metrics["total_context_switches"] += 1

    def _calculate_quantum(self, task: Task) -> float:
        return QUANTUM_BASE * (2 if task.state.dynamic_priority < 8 else 1)

    def run(self, max_time=1000):
        while self.clock < max_time and (self.pending_tasks or self.ready_queue or self.current_task):
            while self.pending_tasks and self.pending_tasks[0][0] <= self.clock:
                _, task = heapq.heappop(self.pending_tasks)
                heapq.heappush(self.ready_queue, task)
                task.state.waiting_time = self.clock - task.config.arrival_time

            if not self.current_task and self.ready_queue:
                self.current_task = heapq.heappop(self.ready_queue)
                if self.current_task.state.start_time < 0:
                    self.current_task.state.start_time = self.clock
                    response_time = self.clock - self.current_task.config.arrival_time
                    self.metrics.task_metrics[self.current_task.config.task_id]["response_time"] = response_time

            if self.current_task:
                quantum = self._calculate_quantum(self.current_task)
                execute_time = min(quantum, self.current_task.state.remaining)

                self.current_task.state.remaining -= execute_time
                self.current_task.state.executed_time += execute_time
                self.total_busy_time += execute_time
                self.clock += execute_time

                for t in self.ready_queue:
                    t.state.waiting_time += execute_time

                if self.current_task.state.remaining <= 0:
                    self.current_task.state.end_time = self.clock
                    turnaround = self.clock - self.current_task.config.arrival_time
                    self.metrics.task_metrics[self.current_task.config.task_id].update({
                        "turnaround_time": turnaround,
                        "waiting_time": self.current_task.state.waiting_time,
                        "executed_time": self.current_task.state.executed_time,
                        "missed_deadline": self.clock > self.current_task.config.deadline
                    })
                    if self.clock > self.current_task.config.deadline:
                        self.metrics.system_metrics["missed_deadlines"] += 1
                    self.current_task = None
                    self.metrics.system_metrics["total_context_switches"] += 1
                else:
                    self._update_priorities()
                    heapq.heappush(self.ready_queue, self.current_task)
                    self.current_task = None

            self._handle_preemption()

        # Final metrics
        execution_times = [t["executed_time"] for t in self.metrics.task_metrics.values()]
        self.metrics.system_metrics.update({
            "cpu_utilization": self.total_busy_time / self.clock if self.clock > 0 else 0,
            "jains_fairness": self.metrics.calculate_jains_fairness(execution_times),
            "avg_waiting_time": np.mean([t["waiting_time"] for t in self.metrics.task_metrics.values()]),
            "avg_response_time": np.mean([t["response_time"] for t in self.metrics.task_metrics.values()])
        })
        return self.metrics
        
# --- Example Usage ---
    # task_id: int
    # arrival_time: float
    # expected_duration: float
    # deadline: float
    # is_io_bound: bool = False
    # is_foreground: bool = False  # OS decides boost based on this

if __name__ == "__main__":
    tasks = [
        # 🟢 I/O-bound + Foreground (should be boosted)
        TaskConfig(1, 0, 10, 20, is_io_bound=True, is_foreground=True),

        # 🔵 CPU-bound + Background (lowest priority)
        TaskConfig(2, 2, 12, 25, is_io_bound=False, is_foreground=False),

        # 🟠 I/O-bound + Background (might get priority for I/O)
        TaskConfig(3, 3, 8, 18, is_io_bound=True, is_foreground=False),

        # 🔴 CPU-bound + Foreground (needs boost for responsiveness)
        TaskConfig(4, 5, 15, 30, is_io_bound=False, is_foreground=True),

        # 🟣 Very short task, late arrival
        TaskConfig(5, 12, 2, 18, is_io_bound=True, is_foreground=True),

        # ⚫ Heavy long CPU-bound background task
        TaskConfig(6, 1, 25, 50, is_io_bound=False, is_foreground=False),

        # 🟡 Mid-range task, edge deadline
        TaskConfig(7, 4, 6, 10, is_io_bound=False, is_foreground=False),

        # 🧠 Foreground but late, I/O heavy
        TaskConfig(8, 14, 5, 20, is_io_bound=True, is_foreground=True),
    ]

    scheduler = Scheduler()
    for t in tasks:
        scheduler.add_task(Task(t))

    metrics = scheduler.run()

    print("\nSystem Metrics:")
    for k, v in metrics.system_metrics.items():
        print(f"{k:>20}: {v:.2f}")

    print("\nPer-Task Metrics:")
    for task_id, data in metrics.task_metrics.items():
        print(f"Task {task_id}:")
        for k, v in data.items():
            print(f"  {k:>16}: {v:.2f}")


System Metrics:
total_context_switches: 8.00
     cpu_utilization: 1.00
    missed_deadlines: 5.00
      jains_fairness: 0.70
    avg_waiting_time: 24.38
   avg_response_time: 17.50

Per-Task Metrics:
Task 1:
     response_time: 0.00
   turnaround_time: 10.00
      waiting_time: 0.00
     executed_time: 10.00
   missed_deadline: 0.00
Task 4:
     response_time: 5.00
   turnaround_time: 35.00
      waiting_time: 20.00
     executed_time: 15.00
   missed_deadline: 1.00
Task 5:
     response_time: 0.00
   turnaround_time: 2.00
      waiting_time: 0.00
     executed_time: 2.00
   missed_deadline: 0.00
Task 8:
     response_time: 0.00
   turnaround_time: 5.00
      waiting_time: 0.00
     executed_time: 5.00
   missed_deadline: 0.00
Task 3:
     response_time: 16.00
   turnaround_time: 24.00
      waiting_time: 16.00
     executed_time: 8.00
   missed_deadline: 1.00
Task 6:
     response_time: 39.00
   turnaround_time: 82.00
      waiting_time: 57.00
     executed_time: 25.00
   missed_dea

# 📅 **Dataset Generation** 

In [26]:
import random
import json
from dataclasses import dataclass
from typing import List, Tuple
from datasets import load_dataset

@dataclass
class TaskConfig:
    task_id: int
    arrival_time: float
    expected_duration: float
    deadline: float
    is_io_bound: bool = False
    is_foreground: bool = False

def generate_random_task(task_id: int, time: float) -> TaskConfig:
    return TaskConfig(
        task_id=task_id,
        arrival_time=time + random.uniform(0, 10),
        expected_duration=random.randint(1, 20),
        deadline=time + random.uniform(15, 60),
        is_io_bound=random.choice([True, False]),
        is_foreground=random.choice([True, False]),
    )

def encode_task(task: TaskConfig) -> List[float]:
    return [
        task.arrival_time,
        task.expected_duration,
        task.deadline,
        1.0 if task.is_io_bound else 0.0,
        1.0 if task.is_foreground else 0.0,
    ]

def heuristic_priority(tasks: List[TaskConfig], strategy="EDF") -> List[int]:
    """Assigns priority 0 (highest) to N-1 (lowest) using a strategy"""
    if strategy == "EDF":
        sorted_tasks = sorted(tasks, key=lambda t: t.deadline)
    elif strategy == "SJF":
        sorted_tasks = sorted(tasks, key=lambda t: t.expected_duration)
    elif strategy == "FCFS":
        sorted_tasks = sorted(tasks, key=lambda t: t.arrival_time)
    else:
        raise ValueError("Unknown strategy")
    
    priorities = [0] * len(tasks)
    for prio, task in enumerate(sorted_tasks):
        priorities[tasks.index(task)] = prio
    return priorities

def generate_dataset_row(tasks: List[TaskConfig], strategy="EDF") -> Tuple[List[List[float]], List[int]]:
    features = [encode_task(t) for t in tasks]
    priorities = heuristic_priority(tasks, strategy)
    return features, priorities

def generate_dataset(num_samples: int = 10000, max_tasks: int = 32, strategy="EDF") -> List[Tuple[List[List[float]], List[int]]]:
    dataset = []
    task_id = 0
    for _ in range(num_samples):
        num_tasks = random.randint(8, max_tasks)  # simulate dynamic task queues
        tasks = [generate_random_task(task_id + i, time=random.uniform(0, 50)) for i in range(num_tasks)]
        task_id += num_tasks
        features, priorities = generate_dataset_row(tasks, strategy)
        dataset.append((features, priorities))
    return dataset

def save_dataset(dataset, path="scheduling_dataset_test_1.json"):
    json_data = [{"input": row[0], "output": row[1]} for row in dataset]
    with open(path, "w") as f:
        json.dump(json_data, f, indent=2)

if __name__ == "__main__":
    data = generate_dataset(num_samples=20000, strategy="EDF")  # Or SJF / FCFS
    save_dataset(data)
    print("Dataset generated and saved.")

# Open the dataset
    
    # Load JSON dataset
    dataset = load_dataset("json", data_files="/kaggle/working/scheduling_dataset_test.json")
    # Check a sample
    print(dataset["train"][0])


KeyboardInterrupt: 

# 🚃 **Training Attention Model**

In [ ]:
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# ------------ Dataset Loader ------------

class TaskPriorityDataset(Dataset):
    def __init__(self, json_file, max_tasks=32):
        with open(json_file, 'r') as f:
            self.data = json.load(f)

        self.max_tasks = max_tasks
        self.feature_dim = 5  # arrival_time, duration, deadline, is_io, is_fg

        self.samples = []
        for row in self.data:
            tasks = row['input']  # shape: (n_tasks, 5)
            order = row['output']  # list of priority indices

            # Create a priority score vector where higher score = higher priority
            # e.g., position 0 in `order` is most important → score = 1.0
            scores = [0.0] * len(tasks)
            for rank, task_idx in enumerate(order):
                scores[task_idx] = 1.0 - (rank / len(order))  # normalized

            # Pad both to max_tasks
            tasks = tasks[:max_tasks] + [[0.0] * self.feature_dim] * (max_tasks - len(tasks))
            scores = scores[:max_tasks] + [0.0] * (max_tasks - len(scores))

            self.samples.append((
                torch.tensor(tasks, dtype=torch.float32),
                torch.tensor(scores, dtype=torch.float32)
            ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


# ------------ Transformer Model ------------

class TaskPriorityTransformer(nn.Module):
    def __init__(self, input_dim=5, model_dim=128, num_heads=8, num_layers=4, max_tasks=32):
        super(TaskPriorityTransformer, self).__init__()
        self.max_tasks = max_tasks
        self.model_dim = model_dim

        self.input_proj = nn.Linear(input_dim, model_dim)
        self.positional_encoding = nn.Parameter(torch.randn(1, max_tasks, model_dim))

        encoder_layer = nn.TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.output_layer = nn.Linear(model_dim, 1)

    def forward(self, x):
        x = self.input_proj(x) + self.positional_encoding[:, :x.size(1), :]
        x = self.transformer(x)
        scores = self.output_layer(x).squeeze(-1)
        return scores  # (batch_size, max_tasks)

# ------------ Training Code ------------

def train_model(json_path, epochs=50, batch_size=64, lr=5e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    dataset = TaskPriorityDataset(json_path)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model = TaskPriorityTransformer().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for batch_inputs, batch_targets in dataloader:
            batch_inputs, batch_targets = batch_inputs.to(device), batch_targets.to(device)

            optimizer.zero_grad()
            outputs = model(batch_inputs)
            loss = criterion(outputs, batch_targets)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

    # Save model
    torch.save(model.state_dict(), "priority_transformer1.pth")
    print("\nModel saved as 'priority_transformer.pth'")

# ------------ Run Training ------------

if __name__ == "__main__":
#     # Ensure your dataset file path is correct
    train_model("/kaggle/working/scheduling_dataset_test.json")


In [ ]:
def infer(model, task_batch, device):
    model.eval()
    with torch.no_grad():
        task_batch = task_batch.to(device)
        outputs = model(task_batch)  # shape: (batch_size, max_tasks)
        preds = torch.argsort(outputs, dim=1, descending=True)  # ranked task indices
    return preds  # (batch_size, max_tasks)

# Evaluate Attention Model

In [ ]:
import heapq
import numpy as np
import torch
import torch.nn as nn
from collections import defaultdict
from torch.utils.data import Dataset
import json

# --- TaskConfig Definition ---
from dataclasses import dataclass

@dataclass
class TaskConfig:
    task_id: int
    arrival_time: float
    expected_duration: float
    deadline: float

# --- Load trained model ---

class TaskPriorityTransformer(nn.Module):
    def __init__(self, input_dim=5, model_dim=128, num_heads=8, num_layers=4, max_tasks=32):
        super(TaskPriorityTransformer, self).__init__()
        self.max_tasks = max_tasks
        # input projection + positional encoding
        self.input_proj = nn.Linear(input_dim, model_dim)
        self.positional_encoding = nn.Parameter(torch.randn(1, max_tasks, model_dim))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=model_dim, nhead=num_heads, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_layer = nn.Linear(model_dim, 1)

    def forward(self, x):
        # x: (batch_size, max_tasks, input_dim)
        x = self.input_proj(x) + self.positional_encoding[:, : x.size(1), :]
        x = self.transformer(x)
        scores = self.output_layer(x).squeeze(-1)
        return scores  # (batch_size, max_tasks)

# --- EDF Scheduler ---

class EDFScheduler:
    def __init__(self):
        self.clock = 0.0
        self.metrics = {
            "missed_deadlines": 0,
            "avg_waiting_time": 0.0,
            "cpu_utilization": 0.0,
        }

    def run(self, tasks):
        total_busy = 0.0
        total_wait = 0.0
        for task in tasks:
            # start immediately at current clock
            start = max(self.clock, task.arrival_time)
            wait = start - task.arrival_time
            end = start + task.expected_duration
            total_busy += task.expected_duration
            total_wait += wait
            if end > task.deadline:
                self.metrics["missed_deadlines"] += 1
            self.clock = end
        n = len(tasks)
        self.metrics["avg_waiting_time"] = total_wait / n if n else 0.0
        self.metrics["cpu_utilization"] = total_busy / self.clock if self.clock > 0.0 else 0.0
        return self.metrics

# --- Dataset ---

class TaskPriorityDataset(Dataset):
    def __init__(self, json_file, max_tasks=32):
        with open(json_file, 'r') as f:
            raw = json.load(f)
        self.max_tasks = max_tasks
        self.feature_dim = 5
        self.samples = []
        for row in raw:
            tasks = row['input']
            # pad or truncate
            padded = tasks[:max_tasks] + [[0.0]*self.feature_dim] * (max_tasks - len(tasks))
            self.samples.append(np.array(padded, dtype=np.float32))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

# --- Evaluation ---

def evaluate_model_vs_edf(model, dataset):
    model.eval()
    sum_edf = defaultdict(float)
    sum_mod = defaultdict(float)
    n = len(dataset)
    with torch.no_grad():
        for idx in range(n):
            tasks = dataset[idx]  # shape (max_tasks, 5)
            # Filter out padded tasks (duration > 0)
            valid = [t for t in tasks if t[1] > 0]
            # Create TaskConfig list
            task_objs = [TaskConfig(i, t[0], t[1], t[2]) for i, t in enumerate(valid)]

            # EDF baseline
            # Sort by deadline
            edf_order = sorted(task_objs, key=lambda t: t.deadline)
            edf_sched = EDFScheduler()
            edf_metrics = edf_sched.run(edf_order)
            for k, v in edf_metrics.items(): sum_edf[k] += v

            # Model-based scheduling
            inp = torch.tensor(tasks).unsqueeze(0)  # (1, max_tasks, 5)
            scores = model(inp).squeeze(0).cpu().numpy()  # (max_tasks,)
            # flatten ensures 1D
            scores = scores.flatten().tolist()
            # pair and sort descending
            paired = [(score, tasks[i]) for i, score in enumerate(scores) if tasks[i][1] > 0]
            paired.sort(key=lambda x: x[0], reverse=True)
            mod_order = [TaskConfig(i, t[0], t[1], t[2]) for i, (_, t) in enumerate(paired)]
            mod_sched = EDFScheduler()
            mod_metrics = mod_sched.run(mod_order)
            for k, v in mod_metrics.items(): sum_mod[k] += v

    # Average metrics
    avg_edf = {k: sum_edf[k]/n for k in sum_edf}
    avg_mod = {k: sum_mod[k]/n for k in sum_mod}

    print("EDF Metrics:")
    for k, v in avg_edf.items(): print(f"{k}: {v:.4f}")
    print("\nModel Metrics:")
    for k, v in avg_mod.items(): print(f"{k}: {v:.4f}")

# --- Main ---

if __name__ == '__main__':
    # load model
    model = TaskPriorityTransformer()
    model.load_state_dict(
        torch.load('/kaggle/working/priority_transformer1.pth', map_location='cpu'),
        strict=True
    )
    # load dataset
    ds = TaskPriorityDataset('scheduling_dataset_test_1.json')
    evaluate_model_vs_edf(model, ds)



# 👑 **TITAN**

**1. Behaviour Cloning**

**2.  Reinforcement Learning**



In [ ]:
import gc
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.distributions import Categorical
import random
from collections import deque
import time

# ------ Data Class Definitions ------
@dataclass
class TaskConfig:
    task_id: int
    arrival_time: float
    expected_duration: float
    deadline: float
    is_io_bound: bool = False
    is_foreground: bool = False

@dataclass
class TaskState:
    remaining: float

@dataclass
class Task:
    config: TaskConfig
    state: TaskState

# ------ Hyperparameters ------
PPO_EPOCHS = 20
CLIP_EPSILON = 0.2
GAMMA = 0.95
ENTROPY_COEF = 0.5
BATCH_SIZE = 512
BC_EPOCHS = 5
NUM_TASKS = 32
BUFFER_SIZE = 10_000
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ------ Transformer SL Model ------
class TaskPriorityTransformer(nn.Module):
    def __init__(self, input_dim=5, model_dim=128, num_heads=8, num_layers=4, max_tasks=32):
        super().__init__()
        self.model_dim = model_dim
        self.input_proj = nn.Linear(input_dim, model_dim)
        self.positional_encoding = nn.Parameter(torch.randn(1, max_tasks, model_dim))
        encoder = nn.TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder, num_layers=num_layers)
        self.output_layer = nn.Linear(model_dim, 1)

    def forward(self, x):
        x = self.input_proj(x)
        x = x + self.positional_encoding[:, :x.size(1), :]
        x = self.transformer(x)
        return self.output_layer(x).squeeze(-1)

# ------ Scheduling Environment ------
class SchedulingEnv:
    def __init__(self, max_tasks=NUM_TASKS):
        self.max_tasks = max_tasks
        self.current_time = 0
        self.tasks = []
        self.reset()

    def reset(self):
        self.current_time = 0
        self.tasks = self._generate_tasks()
        return self._get_state()

    # def _generate_tasks(self):
    #     tasks = []
    #     for i in range(NUM_TASKS):
    #         cfg = TaskConfig(
    #             task_id=i,
    #             arrival_time=random.uniform(0, 100),
    #             expected_duration=random.uniform(1, 100),
    #             deadline=random.uniform(15, 100),
    #             is_io_bound=random.choice([True, False]),
    #             is_foreground=random.choice([True, False])
    #         )
    #         tasks.append(Task(cfg, TaskState(cfg.expected_duration)))
    #     return tasks

    def _generate_tasks(self):
        tasks = []
        for i in range(NUM_TASKS):
            arrival_time = random.expovariate(1 / 10)  # Poisson-like arrivals
            is_io_bound = random.random() < 0.4
            expected_duration = random.uniform(1, 10) if is_io_bound else random.uniform(10, 150)
            slack_factor = random.uniform(1.5, 3.0)
            deadline = arrival_time + expected_duration * slack_factor
            is_foreground = random.random() < 0.3
    
            cfg = TaskConfig(
                task_id=i,
                arrival_time=arrival_time,
                expected_duration=expected_duration,
                deadline=deadline,
                is_io_bound=is_io_bound,
                is_foreground=is_foreground
            )
            tasks.append(Task(cfg, TaskState(cfg.expected_duration)))
        return tasks


    def _get_state(self):
        state = torch.zeros((self.max_tasks, 5), device=DEVICE)
        for i, task in enumerate(self.tasks):
            state[i] = torch.tensor([
                task.config.arrival_time,
                task.config.expected_duration,
                task.config.deadline,
                float(task.config.is_io_bound),
                float(task.config.is_foreground)
            ], device=DEVICE)
        return state

    def step(self, action):
        if action >= len(self.tasks):
            return self._get_state(), -10.0, True, {}
        t = self.tasks[action]
        exec_time = min(t.state.remaining, max(0.1, t.config.deadline - self.current_time))

        t.state.remaining -= exec_time
        self.current_time += exec_time
        reward = self._calc_reward(t, exec_time)
        if t.state.remaining <= 0:
            self.tasks.pop(action)
        done = len(self.tasks) == 0
        return self._get_state(), reward, done, {}

    # def _calc_reward(self, task, dt):
    #     met = (self.current_time + dt) <= task.config.deadline
    #     wait = self.current_time - task.config.arrival_time
    #     r = 4.0 if met else -5.0
    #     r -= 0.1 * wait
    #     r += 1 * dt
    #     r = r/1000.0
    #     return r

    def _calc_reward(self, task, dt):
        cfg = task.config
        state = task.state
    
        # Time-related metrics
        finish_time = self.current_time + dt
        deadline = cfg.deadline
        arrival = cfg.arrival_time
    
        wait_time = self.current_time - arrival
        turnaround_time = finish_time - arrival
        lateness = max(0.0, finish_time - deadline)
    
        # Normalized weights (to avoid huge penalties)
        task_lifetime = max(1.0, deadline - arrival)
        norm_wait = wait_time / task_lifetime
        norm_turnaround = turnaround_time / task_lifetime
        norm_lateness = lateness / task_lifetime
    
        # Base reward: negative lateness, small penalty for wait/turnaround, bonus for execution
        reward = 2 * dt                         # reward progress
        reward -= 2.0 * norm_lateness            # continuous deadline penalty
        reward -= 0.5 * norm_wait                # penalize long waits
        reward -= 0.2 * norm_turnaround          # penalize overall turnaround
    
        # Completion bonus
        if state.remaining <= 0:
            reward += 6.0                        # boost for finishing task
    
        # Scale for stability
        reward = reward / 1000.0
        return reward


# ------ PPO Policy Network ------
class PPOPolicy(nn.Module):
    def __init__(self, sl_model):
        super().__init__()
        self.transformer = sl_model.transformer
        self.input_proj = sl_model.input_proj
        self.pos_encoding = sl_model.positional_encoding
        self.actor = nn.Sequential(
            nn.Linear(sl_model.model_dim, 128), nn.ReLU(), nn.Linear(128, NUM_TASKS)
        )
        self.critic = nn.Sequential(
            nn.Linear(sl_model.model_dim, 128), nn.ReLU(), nn.Linear(128, 1)
        )
        self.optimizer = torch.optim.Adam(self.parameters(), lr=1e-4)
        self.memory = deque(maxlen=BUFFER_SIZE)

    def forward(self, x):
        x_proj = self.input_proj(x)
        x_proj = x_proj + self.pos_encoding[:, :x_proj.size(1), :]
        out = self.transformer(x_proj)
        mask = (x[:, :, 0] != 0).float().unsqueeze(-1)
        feats = (out * mask).mean(dim=1)
        logits = self.actor(feats)
        value = self.critic(feats).squeeze(-1)
        return logits, value

    def collect_trajectories(self, env, episodes=10):
        for ep in range(episodes):
            state = env.reset()
            done = False
            total_r = 0.0
            while not done:
                with torch.no_grad():
                    logits, value = self.forward(state.unsqueeze(0))
                    valid = (state[:, 0] != 0)
                    logits[0, ~valid] = -1e9
                    dist = Categorical(logits=logits)
                    action = dist.sample().item()
                    lp = dist.log_prob(torch.tensor(action, device=DEVICE))
    
                    next_state, reward, done, _ = env.step(action)
                    total_r += reward
                    with torch.no_grad():
                        _, next_val = self.forward(next_state.unsqueeze(0))
    
                    self.memory.append((
                        state.cpu(), action, reward, done,
                        lp.detach().cpu(), value.detach().cpu(), next_val.detach().cpu()
                    ))
                    state = next_state
            print(f"[Collect] Episode {ep+1}/{episodes}, Return: {total_r:.2f}")

    def train(self):
        if len(self.memory) < BATCH_SIZE:
            print(f"[Train] Not enough samples: {len(self.memory)}/{BATCH_SIZE}")
            return
        idxs = random.sample(range(len(self.memory)), BATCH_SIZE)
        batch = [self.memory[i] for i in idxs]
        states, actions, rewards, dones, old_lps, vals, next_vals = zip(*batch)

        states = torch.stack(states).to(DEVICE)
        actions = torch.tensor(actions, device=DEVICE)
        rewards = torch.tensor(rewards, device=DEVICE)
        dones   = torch.tensor(dones, dtype=torch.float32, device=DEVICE)
        old_lps = torch.stack(old_lps).to(DEVICE)
        vals    = torch.stack(vals).to(DEVICE)
        next_vals = torch.stack(next_vals).to(DEVICE)

        returns = rewards + GAMMA * next_vals * (1 - dones)
        advantages = returns - vals

        for epoch in range(PPO_EPOCHS):
            logits, values = self.forward(states)
            dist = Categorical(logits=logits)
            new_lps = dist.log_prob(actions)
            ratio = (new_lps - old_lps).exp()

            s1 = ratio * advantages
            s2 = torch.clamp(ratio, 1-CLIP_EPSILON, 1+CLIP_EPSILON) * advantages
            pol_loss = -torch.min(s1, s2).mean()
            val_loss = 0.5 * (values - returns).pow(2).mean()
            ent_loss = -ENTROPY_COEF * dist.entropy().mean()
            total_loss = pol_loss + val_loss + ent_loss

            self.optimizer.zero_grad()
            total_loss.backward()
            self.optimizer.step()

            print(f"[PPO] Epoch {epoch+1}/{PPO_EPOCHS} — Policy: {pol_loss.item():.4f}, Value: {val_loss.item():.4f}, Entropy: {ent_loss.item():.4f}")

        self.memory.clear()
        gc.collect()
        print("[Train] Buffer cleared.")

# ------ Behavior Cloning ------
# def behavior_cloning(model, env, epochs=BC_EPOCHS):
#     opt = torch.optim.Adam(model.parameters())
#     for ep in range(epochs):
#         epoch_start = time.time()
#         state = env.reset()
#         states, actions = [], []
#         done = False
#         while not done:
#             with torch.no_grad():
#                 logits, _ = model(state.unsqueeze(0))
#                 mask = (state[:,0]!=0)
#                 logits[0,~mask] = -1e9
#                 action = torch.argmax(logits[0]).item()

#             states.append(state)
#             actions.append(action)
#             state, _, done, _ = env.step(action)

#         S = torch.stack(states).to(DEVICE)
#         A = torch.tensor(actions, device=DEVICE)
#         for i in range(20):
#             logits, _ = model(S)
#             loss = nn.CrossEntropyLoss()(logits, A)
#             opt.zero_grad(); loss.backward(); opt.step()
#             print(f"[BC] Epoch {ep+1}/{epochs}, Step {i+1}/20 — Loss: {loss.item():.4f}")
#         duration = time.time() - epoch_start
#         print(f"[BC] Finished epoch {ep+1}/{epochs} in {duration:.2f}s")

def behavior_cloning(model, env, epochs=BC_EPOCHS):
    opt = torch.optim.Adam(model.parameters())
    batch_size = 64  # Adjusted batch size to manage GPU memory

    for ep in range(epochs):
        epoch_start = time.time()
        state = env.reset()
        states, actions = [], []
        done = False
        
        # Collect trajectory
        while not done:
            with torch.no_grad():
                logits, _ = model(state.unsqueeze(0))
                mask = (state[:, 0] != 0)
                logits[0, ~mask] = -1e9
                action = torch.argmax(logits[0]).item()
            
            states.append(state)
            actions.append(action)
            state, _, done, _ = env.step(action)
        
        # Convert to tensors and move to device in batches
        S = torch.stack(states).to(DEVICE)
        A = torch.tensor(actions, device=DEVICE)
        
        # Create DataLoader for mini-batch training
        dataset = torch.utils.data.TensorDataset(S, A)
        dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
        
        # Train with mini-batches
        for i in range(20):  # 20 optimization steps per epoch
            total_loss = 0.0
            for s_batch, a_batch in dataloader:
                logits, _ = model(s_batch)
                loss = nn.CrossEntropyLoss()(logits, a_batch)
                
                opt.zero_grad()
                loss.backward()
                opt.step()
                
                total_loss += loss.item()
            
            avg_loss = total_loss / len(dataloader)
            print(f"[BC] Epoch {ep+1}/{epochs}, Step {i+1}/20 — Loss: {avg_loss:.4f}")
        
        duration = time.time() - epoch_start
        print(f"[BC] Finished epoch {ep+1}/{epochs} in {duration:.2f}s")

# ------ Main ------
if __name__ == '__main__':
    sl = TaskPriorityTransformer().to(DEVICE)
    sl.load_state_dict(torch.load(
        '/kaggle/input/sgpt/pytorch/default/1/priority_transformer1 (1).pth', map_location=DEVICE
    ))
    env = SchedulingEnv()
    policy = PPOPolicy(sl).to(DEVICE)
    
    print('Behavior Cloning...')
    behavior_cloning(policy, env)
    print('BC Done.')

    print('PPO Training...')
    for i in range(1, 50):
        print(f"--- Update {i}/100 ---")
        policy.collect_trajectories(env)
        policy.train()
    torch.save(policy.state_dict(), 'rl_scheduler_efficient_1.pth')


# **EVALUATION**

In [ ]:
import random
import torch
import torch.nn as nn
from torch.distributions import Categorical
from collections import deque
from dataclasses import dataclass

# ------ Hyperparameters ------
PPO_EPOCHS    = 10
CLIP_EPSILON  = 0.1
GAMMA         = 0.95
ENTROPY_COEF  = 0.2
BATCH_SIZE    = 256
BC_EPOCHS     = 10
NUM_TASKS     = 32
BUFFER_SIZE   = 10_000
DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ------ Data Classes ------
@dataclass
class TaskConfig:
    task_id: int
    arrival_time: float
    expected_duration: float
    deadline: float
    is_io_bound: bool = False
    is_foreground: bool = False

@dataclass
class TaskState:
    remaining: float

@dataclass
class Task:
    config: TaskConfig
    state: TaskState

# ------ Scheduling Environment ------
class SchedulingEnv:
    def __init__(self, max_tasks=NUM_TASKS):
        self.max_tasks = max_tasks
        self.current_time = 0.0
        self.tasks = []
        self.reset()

    def reset(self):
        self.current_time = 0.0
        self.tasks = self._generate_tasks()
        return self._get_state()

    def _generate_tasks(self):
        tasks = []
        for i in range(NUM_TASKS):
            cfg = TaskConfig(
                task_id=i,
                arrival_time=random.uniform(0, 10),
                expected_duration=random.uniform(1, 20),
                deadline=random.uniform(15, 60),
                is_io_bound=random.choice([True, False]),
                is_foreground=random.choice([True, False])
            )
            tasks.append(Task(cfg, TaskState(cfg.expected_duration)))
        return tasks

    def _get_state(self):
        state = torch.zeros((self.max_tasks, 5), device=DEVICE)
        for i, t in enumerate(self.tasks):
            state[i] = torch.tensor([
                t.config.arrival_time,
                t.config.expected_duration,
                t.config.deadline,
                float(t.config.is_io_bound),
                float(t.config.is_foreground)
            ], device=DEVICE)
        return state

    def step(self, action):
        if action >= len(self.tasks):
            return self._get_state(), -10.0, True, {}
        t = self.tasks[action]
        dt = min(t.state.remaining, max(0.1, t.config.deadline - self.current_time))
        t.state.remaining -= dt
        self.current_time     += dt
        # reward scaled
        r = (4.0 if (self.current_time <= t.config.deadline) else -5.0)
        r -= 0.1 * (self.current_time - t.config.arrival_time)
        r += 1.0 * dt
        r /= 1000.0
        if t.state.remaining <= 0:
            self.tasks.pop(action)
        done = (len(self.tasks)==0)
        return self._get_state(), r, done, {}

# ------ Models ------
class TaskPriorityTransformer(nn.Module):
    def __init__(self, input_dim=5, model_dim=128, num_heads=8, num_layers=4, max_tasks=NUM_TASKS):
        super().__init__()
        self.model_dim = model_dim
        self.input_proj = nn.Linear(input_dim, model_dim)
        self.positional_encoding = nn.Parameter(torch.randn(1, max_tasks, model_dim))
        encoder = nn.TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder, num_layers=num_layers)
        self.output_layer = nn.Linear(model_dim, 1)

    def forward(self, x):
        x = self.input_proj(x) + self.positional_encoding[:, :x.size(1), :]
        x = self.transformer(x)
        return self.output_layer(x).squeeze(-1)

class PPOPolicy(nn.Module):
    def __init__(self, sl_model):
        super().__init__()
        # reuse SL model components
        self.transformer = sl_model.transformer
        self.input_proj  = sl_model.input_proj
        self.positional_encoding = sl_model.positional_encoding
        md = sl_model.model_dim
        self.actor  = nn.Sequential(nn.Linear(md,128), nn.ReLU(), nn.Linear(128, NUM_TASKS))
        self.critic = nn.Sequential(nn.Linear(md,128), nn.ReLU(), nn.Linear(128, 1))
        self.optimizer = torch.optim.Adam(self.parameters(), lr=1e-4)
        self.memory    = deque(maxlen=BUFFER_SIZE)

    def forward(self, x):
        x = self.input_proj(x) + self.positional_encoding[:, :x.size(1), :]
        h = self.transformer(x)
        mask = (x[:,:,0]!=0).float().unsqueeze(-1)
        feats = (h * mask).mean(dim=1)
        return self.actor(feats), self.critic(feats).squeeze(-1)

    def collect_trajectories(self, env, episodes=10):
        for ep in range(episodes):
            s, done, R = env.reset(), False, 0.0
            while not done:
                logits, _ = self.forward(s.unsqueeze(0))
                logits[0, ~(s[:,0]!=0)] = -1e9
                dist = Categorical(logits=logits)
                a = dist.sample().item()
                s, r, done, _ = env.step(a)
                R += r
            print(f"[Collect] Ep{ep+1}/{episodes}, Return={R:.3f}")

    def update_policy(self):
        if len(self.memory) < BATCH_SIZE:
            return
        batch = random.sample(self.memory, BATCH_SIZE)
        s,a,r,d,lp,vs,nv = zip(*batch)
        S = torch.stack(s).to(DEVICE)
        A = torch.tensor(a,device=DEVICE)
        R = torch.tensor(r,device=DEVICE)
        D = torch.tensor(d,dtype=torch.float,device=DEVICE)
        LP= torch.stack(lp).to(DEVICE)
        V = torch.stack(vs).to(DEVICE)
        NV= torch.stack(nv).to(DEVICE)

        ret = R + GAMMA * NV * (1-D)
        adv = ret - V

        for _ in range(PPO_EPOCHS):
            logits, vals = self.forward(S)
            dist = Categorical(logits=logits)
            new_lp = dist.log_prob(A)
            ratio = (new_lp - LP).exp()

            s1 = ratio * adv
            s2 = torch.clamp(ratio,1-CLIP_EPSILON,1+CLIP_EPSILON) * adv
            pol_loss = -torch.min(s1,s2).mean()
            val_loss = 0.5*(vals.squeeze()-ret).pow(2).mean()
            ent_loss = -ENTROPY_COEF * dist.entropy().mean()

            self.optimizer.zero_grad()
            (pol_loss + val_loss + ent_loss).backward()
            self.optimizer.step()

        self.memory.clear()

# ------ FixedTaskEnv------------ ------
class FixedTaskEnv(SchedulingEnv):
    def __init__(self, task_list):
        self.original_tasks = task_list
        self.max_tasks      = len(task_list)
        self.current_time   = 0.0
        self.tasks          = []

    def reset(self):
        self.current_time = 0.0
        # deep-copy task list
        self.tasks = [Task(TaskConfig(**vars(t.config)), TaskState(t.state.remaining))
                      for t in self.original_tasks]
        return self._get_state()

# ------ Inference & Evaluation ------
def edf_heuristic(state):
    mask = (state[:,0]!=0)
    dln  = state[:,2].clone()
    dln[~mask]=float('inf')
    return torch.argmin(dln).item()

def infer_sl(sl, s):
    sl.eval()
    with torch.no_grad():
        lg = sl(s.unsqueeze(0))[0]
        lg[~(s[:,0]!=0)] = -1e9
        return torch.argmax(lg).item()

def infer_rl(rl, s):
    rl.eval()
    with torch.no_grad():
        lg,_ = rl(s.unsqueeze(0))
        lg[0,~(s[:,0]!=0)] = -1e9
        return Categorical(logits=lg).sample().item()

# ------ Dataset & Compare ------
def generate_task_list(seed):
    random.seed(seed)
    return SchedulingEnv()._generate_tasks()

def evaluate(sl, rl, episodes=10, seed0=123):
    res=[]
    for i in range(episodes):
        tasks = generate_task_list(seed0+i)
        e1,e2,e3 = FixedTaskEnv(tasks), FixedTaskEnv(tasks), FixedTaskEnv(tasks)
        R1=R2=R3=0.0
        s=e1.reset()
        while True:
            a = edf_heuristic(s)
            s,r,done,_ = e1.step(a)
            R1+=r
            if done: break
        s=e2.reset()
        while True:
            a = infer_sl(sl,s)
            s,r,done,_ = e2.step(a)
            R2+=r
            if done: break
        s=e3.reset()
        while True:
            a = infer_rl(rl,s)
            s,r,done,_ = e3.step(a)
            R3+=r
            if done: break
        res.append({'episode':i+1,'EDF':R1,'SL':R2,'RL':R3})
    return res

# ------ Main ------
if __name__=='__main__':
    # Load supervised SL model
    sl = TaskPriorityTransformer().to(DEVICE)
    state_dict_sl = torch.load(
        '/kaggle/input/sgpt/pytorch/default/1/priority_transformer1 (1).pth',
        map_location=DEVICE
    )
    sl.load_state_dict(state_dict_sl)

    # Load RL policy
    rl = PPOPolicy(sl).to(DEVICE)
    state_dict_rl = torch.load(
        '/kaggle/input/schedulegpt/pytorch/default/1/rl_scheduler_efficient.pth',
        map_location=DEVICE
    )
    # fix key mismatch: rename 'pos_encoding' keys to 'positional_encoding'
    fixed_state = {}
    for k, v in state_dict_rl.items():
        new_key = k.replace('pos_encoding', 'positional_encoding')
        fixed_state[new_key] = v
    rl.load_state_dict(fixed_state)

    # # Evaluate policies
    # results = evaluate(sl, rl, episodes=10)
    # for row in results:
    #     print(row)

def evaluate_metrics(policy_fn, task_list):
    """
    policy_fn: (env, state) -> action_index
    Returns avg_turnaround, avg_waiting, miss_rate, avg_response
    """
    env = FixedTaskEnv(task_list)
    state = env.reset()

    # Initialize tracking
    all_ids    = [t.config.task_id for t in task_list]
    arr_time   = {t.config.task_id: t.config.arrival_time for t in task_list}
    deadline   = {t.config.task_id: t.config.deadline     for t in task_list}
    start_time = {tid: None for tid in all_ids}
    comp_time  = {}

    # We'll compare ID-lists before & after each step
    remaining_ids = set(all_ids)

    while remaining_ids:
        # 1) choose an action
        action = policy_fn(env, state)

        # 2) grab the task ID and mark response
        tid = env.tasks[action].config.task_id
        if start_time[tid] is None:
            start_time[tid] = env.current_time

        # 3) snapshot IDs before and after stepping
        before_ids = [t.config.task_id for t in env.tasks]
        state, _, done, _ = env.step(action)
        after_ids  = [t.config.task_id for t in env.tasks]

        # 4) if tid vanished, record completion
        if tid in before_ids and tid not in after_ids:
            comp_time[tid] = env.current_time
            remaining_ids.remove(tid)

        if done:
            break

    N = len(all_ids)
    # Turnaround = completion - arrival
    tat  = [comp_time[tid] - arr_time[tid] for tid in all_ids]
    # Waiting    = start     - arrival
    wait = [start_time[tid] - arr_time[tid] for tid in all_ids]
    # Deadline misses
    misses = sum(1 for tid in all_ids if comp_time[tid] > deadline[tid])

    return {
      'avg_turnaround': sum(tat)/N,
      'avg_waiting'   : sum(wait)/N,
      'miss_rate'     : misses/N,
      'avg_response'  : sum(start_time[tid]-arr_time[tid] for tid in all_ids)/N
    }

for ep in range(100):
    seed      = 100 + ep
    task_list = generate_task_list(seed)

    m_edf = evaluate_metrics(lambda e,s: edf_heuristic(s),   task_list)
    m_sl  = evaluate_metrics(lambda e,s: infer_sl(sl, s),   task_list)
    m_rl  = evaluate_metrics(lambda e,s: infer_rl(rl, s),   task_list)

    print(f"Episode {ep+1}:")
    print(" EDF:", m_edf)
    print(" SL :", m_sl)
    print(" RL :", m_rl)


# **Standalone Script For Infering TITAN**

In [ ]:
import random
import torch
import torch.nn as nn
from torch.distributions import Categorical
from collections import deque
from dataclasses import dataclass
import pandas as pd

# ------ Hyperparameters ------
PPO_EPOCHS    = 10
CLIP_EPSILON  = 0.1
GAMMA         = 0.95
ENTROPY_COEF  = 0.2
BATCH_SIZE    = 256
BC_EPOCHS     = 10
NUM_TASKS     = 32
BUFFER_SIZE   = 10_000
DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ------ Data Classes ------
@dataclass
class TaskConfig:
    task_id: int
    arrival_time: float
    expected_duration: float
    deadline: float
    is_io_bound: bool = False
    is_foreground: bool = False

@dataclass
class TaskState:
    remaining: float

@dataclass
class Task:
    config: TaskConfig
    state: TaskState

# ------ Scheduling Environment ------
class SchedulingEnv:
    def __init__(self, max_tasks=NUM_TASKS):
        self.max_tasks = max_tasks
        self.current_time = 0.0
        self.tasks = []
        self.reset()

    def reset(self):
        self.current_time = 0.0
        self.tasks = self._generate_tasks()
        return self._get_state()

    def _generate_tasks(self):
        tasks = []
        for i in range(NUM_TASKS):
            cfg = TaskConfig(
                task_id=i,
                arrival_time=random.uniform(0, 10),
                expected_duration=random.uniform(1, 20),
                deadline=random.uniform(15, 60),
                is_io_bound=random.choice([True, False]),
                is_foreground=random.choice([True, False])
            )
            tasks.append(Task(cfg, TaskState(cfg.expected_duration)))
        return tasks

    def _get_state(self):
        state = torch.zeros((self.max_tasks, 5), device=DEVICE)
        for i, t in enumerate(self.tasks):
            state[i] = torch.tensor([
                t.config.arrival_time,
                t.config.expected_duration,
                t.config.deadline,
                float(t.config.is_io_bound),
                float(t.config.is_foreground)
            ], device=DEVICE)
        return state

    def step(self, action):
        if action >= len(self.tasks):
            return self._get_state(), -10.0, True, {}
        t = self.tasks[action]
        dt = min(t.state.remaining, max(0.1, t.config.deadline - self.current_time))
        t.state.remaining -= dt
        self.current_time += dt
        # reward scaled
        r = (4.0 if (self.current_time <= t.config.deadline) else -5.0)
        r -= 0.1 * (self.current_time - t.config.arrival_time)
        r += 1.0 * dt
        r /= 1000.0
        if t.state.remaining <= 0:
            self.tasks.pop(action)
        done = (len(self.tasks)==0)
        return self._get_state(), r, done, {}

# ------ Models ------
class TaskPriorityTransformer(nn.Module):
    def __init__(self, input_dim=5, model_dim=128, num_heads=8, num_layers=4, max_tasks=NUM_TASKS):
        super().__init__()
        self.model_dim = model_dim
        self.input_proj = nn.Linear(input_dim, model_dim)
        self.positional_encoding = nn.Parameter(torch.randn(1, max_tasks, model_dim))
        encoder = nn.TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder, num_layers=num_layers)
        self.output_layer = nn.Linear(model_dim, 1)

    def forward(self, x):
        x = self.input_proj(x) + self.positional_encoding[:, :x.size(1), :]
        x = self.transformer(x)
        return self.output_layer(x).squeeze(-1)

class PPOPolicy(nn.Module):
    def __init__(self, sl_model):
        super().__init__()
        # reuse SL model components
        self.transformer = sl_model.transformer
        self.input_proj  = sl_model.input_proj
        self.positional_encoding = sl_model.positional_encoding
        md = sl_model.model_dim
        self.actor  = nn.Sequential(nn.Linear(md,128), nn.ReLU(), nn.Linear(128, NUM_TASKS))
        self.critic = nn.Sequential(nn.Linear(md,128), nn.ReLU(), nn.Linear(128, 1))
        self.optimizer = torch.optim.Adam(self.parameters(), lr=1e-4)
        self.memory    = deque(maxlen=BUFFER_SIZE)

    def forward(self, x):
        x = self.input_proj(x) + self.positional_encoding[:, :x.size(1), :]
        h = self.transformer(x)
        mask = (x[:,:,0]!=0).float().unsqueeze(-1)
        feats = (h * mask).mean(dim=1)
        return self.actor(feats), self.critic(feats).squeeze(-1)

# ------ FixedTaskEnv ------
class FixedTaskEnv(SchedulingEnv):
    def __init__(self, task_list):
        self.original_tasks = task_list
        self.max_tasks = len(task_list)
        self.current_time = 0.0
        self.tasks = []

    def reset(self):
        self.current_time = 0.0
        # deep-copy task list
        self.tasks = [Task(TaskConfig(**vars(t.config)), TaskState(t.state.remaining))
                      for t in self.original_tasks]
        return self._get_state()

# ------ Inference Functions ------
def edf_heuristic(state):
    mask = (state[:,0]!=0)
    dln = state[:,2].clone()
    dln[~mask] = float('inf')
    return torch.argmin(dln).item()

def infer_sl(sl, s):
    sl.eval()
    with torch.no_grad():
        lg = sl(s.unsqueeze(0))[0]
        lg[~(s[:,0]!=0)] = -1e9
        return torch.argmax(lg).item()

def infer_rl(rl, s):
    rl.eval()
    with torch.no_grad():
        lg, _ = rl(s.unsqueeze(0))
        lg[0,~(s[:,0]!=0)] = -1e9
        return Categorical(logits=lg).sample().item()

# Define Titan as the RL model with a specific temperature parameter
def infer_titan(rl, s):
    rl.eval()
    with torch.no_grad():
        lg, _ = rl(s.unsqueeze(0))
        lg[0,~(s[:,0]!=0)] = -1e9
        
        # Use a different temperature for Titan to differentiate from regular RL
        temperature = 0.75  # Lower temperature for more exploitation
        probs = torch.softmax(lg[0] / temperature, dim=0)
        return torch.multinomial(probs, 1).item()

# ------ Dataset Generation ------
def generate_task_list(seed):
    random.seed(seed)
    return SchedulingEnv()._generate_tasks()

# ------ Evaluation Metrics ------
def evaluate_metrics(policy_fn, task_list):
    """
    policy_fn: (env, state) -> action_index
    Returns avg_turnaround, avg_waiting, miss_rate, avg_response
    """
    env = FixedTaskEnv(task_list)
    state = env.reset()
    # Initialize tracking
    all_ids = [t.config.task_id for t in task_list]
    arr_time = {t.config.task_id: t.config.arrival_time for t in task_list}
    deadline = {t.config.task_id: t.config.deadline for t in task_list}
    start_time = {tid: None for tid in all_ids}
    comp_time = {}
    # We'll compare ID-lists before & after each step
    remaining_ids = set(all_ids)
    while remaining_ids:
        # 1) choose an action
        action = policy_fn(env, state)
        # 2) grab the task ID and mark response
        tid = env.tasks[action].config.task_id
        if start_time[tid] is None:
            start_time[tid] = env.current_time
        # 3) snapshot IDs before and after stepping
        before_ids = [t.config.task_id for t in env.tasks]
        state, _, done, _ = env.step(action)
        after_ids = [t.config.task_id for t in env.tasks]
        # 4) if tid vanished, record completion
        if tid in before_ids and tid not in after_ids:
            comp_time[tid] = env.current_time
            remaining_ids.remove(tid)
        if done:
            break
    N = len(all_ids)
    # Turnaround = completion - arrival
    tat = [comp_time.get(tid, env.current_time) - arr_time[tid] for tid in all_ids]
    # Waiting = start - arrival
    wait = [start_time.get(tid, env.current_time) - arr_time[tid] for tid in all_ids]
    # Deadline misses
    misses = sum(1 for tid in all_ids if comp_time.get(tid, float('inf')) > deadline[tid])
    return {
      'avg_turnaround': sum(tat)/N,
      'avg_waiting': sum(wait)/N,
      'miss_rate': misses/N,
      'avg_response': sum(start_time.get(tid, env.current_time)-arr_time[tid] for tid in all_ids)/N
    }

# ------ Main Function ------
def main():
    # Load supervised SL model
    sl = TaskPriorityTransformer().to(DEVICE)
    try:
        state_dict_sl = torch.load(
            '/kaggle/input/sgpt/pytorch/default/1/priority_transformer1 (1).pth',
            map_location=DEVICE
        )
        sl.load_state_dict(state_dict_sl)
        print("Loaded SL model successfully.")
    except Exception as e:
        print(f"Could not load SL model: {e}")
    
    # Load RL policy (also used as Titan with different inference strategy)
    rl = PPOPolicy(sl).to(DEVICE)
    try:
        state_dict_rl = torch.load(
            '/kaggle/input/titan-scheduler/pytorch/default/1/rl_scheduler_efficient.pth',
            map_location=DEVICE
        )
        # Fix key mismatch: rename 'pos_encoding' keys to 'positional_encoding'
        fixed_state = {}
        for k, v in state_dict_rl.items():
            new_key = k.replace('pos_encoding', 'positional_encoding')
            fixed_state[new_key] = v
        rl.load_state_dict(fixed_state)
        print("Loaded RL model successfully.")
    except Exception as e:
        print(f"Could not load RL model: {e}")
    
    # Create a results dictionary to store all metrics
    all_results = []
    
    # Run the evaluation
    for ep in range(100):
        seed = 100 + ep
        task_list = generate_task_list(seed)
        
        # Evaluate each policy on the same task list
        m_edf = evaluate_metrics(lambda e, s: edf_heuristic(s), task_list)
        m_sl = evaluate_metrics(lambda e, s: infer_sl(sl, s), task_list)
        m_titan = evaluate_metrics(lambda e, s: infer_titan(rl, s), task_list)
        
        # Store results
        episode_results = {
            'episode': ep + 1,
            'edf_turnaround': m_edf['avg_turnaround'],
            'edf_waiting': m_edf['avg_waiting'],
            'edf_miss_rate': m_edf['miss_rate'],
            'edf_response': m_edf['avg_response'],
            'sl_turnaround': m_sl['avg_turnaround'],
            'sl_waiting': m_sl['avg_waiting'],
            'sl_miss_rate': m_sl['miss_rate'],
            'sl_response': m_sl['avg_response'],
            'titan_turnaround': m_titan['avg_turnaround'],
            'titan_waiting': m_titan['avg_waiting'],
            'titan_miss_rate': m_titan['miss_rate'],
            'titan_response': m_titan['avg_response']
        }
        all_results.append(episode_results)
        
        # Print episode results
        print(f"Episode {ep+1}:")
        print(" EDF:", m_edf)
        print(" SL :", m_sl)
        print(" TITAN:", m_titan)
    
    # Convert results to DataFrame and save to CSV
    results_df = pd.DataFrame(all_results)
    results_df.to_csv('scheduler_evaluation_results.csv', index=False)
    
    # Print average metrics
    print("\nAverage Metrics:")
    avg_metrics = results_df.mean()
    for col in results_df.columns:
        if col != 'episode':
            print(f"{col}: {avg_metrics[col]:.4f}")

if __name__ == '__main__':
    main()

In [27]:
import random
import torch
import torch.nn as nn
from torch.distributions import Categorical
from collections import deque
from dataclasses import dataclass
import pandas as pd

# ------ Hyperparameters ------
PPO_EPOCHS    = 10
CLIP_EPSILON  = 0.1
GAMMA         = 0.95
ENTROPY_COEF  = 0.2
BATCH_SIZE    = 256
BC_EPOCHS     = 10
NUM_TASKS     = 32
BUFFER_SIZE   = 10_000
DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ------ Data Classes ------
@dataclass
class TaskConfig:
    task_id: int
    arrival_time: float
    expected_duration: float
    deadline: float
    is_io_bound: bool = False
    is_foreground: bool = False

@dataclass
class TaskState:
    remaining: float

@dataclass
class Task:
    config: TaskConfig
    state: TaskState

# ------ Scheduling Environment ------
class SchedulingEnv:
    def __init__(self, max_tasks=NUM_TASKS):
        self.max_tasks = max_tasks
        self.current_time = 0.0
        self.tasks = []
        self.reset()

    def reset(self):
        self.current_time = 0.0
        self.tasks = self._generate_tasks()
        return self._get_state()

    def _generate_tasks(self):
        tasks = []
        for i in range(NUM_TASKS):
            cfg = TaskConfig(
                task_id=i,
                arrival_time=random.uniform(0, 10),
                expected_duration=random.uniform(1, 20),
                deadline=random.uniform(15, 60),
                is_io_bound=random.choice([True, False]),
                is_foreground=random.choice([True, False])
            )
            tasks.append(Task(cfg, TaskState(cfg.expected_duration)))
        return tasks

    def _get_state(self):
        state = torch.zeros((self.max_tasks, 5), device=DEVICE)
        for i, t in enumerate(self.tasks):
            state[i] = torch.tensor([
                t.config.arrival_time,
                t.config.expected_duration,
                t.config.deadline,
                float(t.config.is_io_bound),
                float(t.config.is_foreground)
            ], device=DEVICE)
        return state

    def step(self, action):
        if action >= len(self.tasks):
            return self._get_state(), -10.0, True, {}
        t = self.tasks[action]
        dt = min(t.state.remaining, max(0.1, t.config.deadline - self.current_time))
        t.state.remaining -= dt
        self.current_time += dt
        # reward scaled
        r = (4.0 if (self.current_time <= t.config.deadline) else -5.0)
        r -= 0.1 * (self.current_time - t.config.arrival_time)
        r += 1.0 * dt
        r /= 1000.0
        if t.state.remaining <= 0:
            self.tasks.pop(action)
        done = (len(self.tasks)==0)
        return self._get_state(), r, done, {}

# ------ Models ------
class TaskPriorityTransformer(nn.Module):
    def __init__(self, input_dim=5, model_dim=128, num_heads=8, num_layers=4, max_tasks=NUM_TASKS):
        super().__init__()
        self.model_dim = model_dim
        self.input_proj = nn.Linear(input_dim, model_dim)
        self.positional_encoding = nn.Parameter(torch.randn(1, max_tasks, model_dim))
        encoder = nn.TransformerEncoderLayer(d_model=model_dim, nhead=num_heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder, num_layers=num_layers)
        self.output_layer = nn.Linear(model_dim, 1)

    def forward(self, x):
        x = self.input_proj(x) + self.positional_encoding[:, :x.size(1), :]
        x = self.transformer(x)
        return self.output_layer(x).squeeze(-1)

class PPOPolicy(nn.Module):
    def __init__(self, sl_model):
        super().__init__()
        # reuse SL model components
        self.transformer = sl_model.transformer
        self.input_proj  = sl_model.input_proj
        self.positional_encoding = sl_model.positional_encoding
        md = sl_model.model_dim
        self.actor  = nn.Sequential(nn.Linear(md,128), nn.ReLU(), nn.Linear(128, NUM_TASKS))
        self.critic = nn.Sequential(nn.Linear(md,128), nn.ReLU(), nn.Linear(128, 1))
        self.optimizer = torch.optim.Adam(self.parameters(), lr=1e-4)
        self.memory    = deque(maxlen=BUFFER_SIZE)

    def forward(self, x):
        x = self.input_proj(x) + self.positional_encoding[:, :x.size(1), :]
        h = self.transformer(x)
        mask = (x[:,:,0]!=0).float().unsqueeze(-1)
        feats = (h * mask).mean(dim=1)
        return self.actor(feats), self.critic(feats).squeeze(-1)

# ------ FixedTaskEnv ------
class FixedTaskEnv(SchedulingEnv):
    def __init__(self, task_list):
        self.original_tasks = task_list
        self.max_tasks = len(task_list)
        self.current_time = 0.0
        self.tasks = []

    def reset(self):
        self.current_time = 0.0
        # deep-copy task list
        self.tasks = [Task(TaskConfig(**vars(t.config)), TaskState(t.state.remaining))
                      for t in self.original_tasks]
        return self._get_state()

# ------ Inference Functions ------
def edf_heuristic(state):
    mask = (state[:,0]!=0)
    dln = state[:,2].clone()
    dln[~mask] = float('inf')
    return torch.argmin(dln).item()

def infer_sl(sl, s):
    sl.eval()
    with torch.no_grad():
        lg = sl(s.unsqueeze(0))[0]
        lg[~(s[:,0]!=0)] = -1e9
        return torch.argmax(lg).item()

def infer_rl(rl, s):
    rl.eval()
    with torch.no_grad():
        lg, _ = rl(s.unsqueeze(0))
        lg[0,~(s[:,0]!=0)] = -1e9
        return Categorical(logits=lg).sample().item()

# Define Titan as the RL model with a specific temperature parameter
def infer_titan(rl, s):
    rl.eval()
    with torch.no_grad():
        lg, _ = rl(s.unsqueeze(0))
        lg[0,~(s[:,0]!=0)] = -1e9
        
        # Use a different temperature for Titan to differentiate from regular RL
        temperature = 0.75  # Lower temperature for more exploitation
        probs = torch.softmax(lg[0] / temperature, dim=0)
        return torch.multinomial(probs, 1).item()

# ------ Dataset Generation ------
def generate_task_list(seed):
    random.seed(seed)
    return SchedulingEnv()._generate_tasks()

# ------ Evaluation Metrics ------
# Replace the original `evaluate_metrics` function with this updated version:
def evaluate_metrics(policy_fn, task_list):
    """
    policy_fn: (env, state) -> action_index
    Returns dictionary of metrics including turnaround, waiting, miss_rate, response,
    context switches, CPU utilization, throughput, makespan, CPU idle time, average burst, and preemptions.
    """
    env = FixedTaskEnv(task_list)
    state = env.reset()

    all_ids = [t.config.task_id for t in task_list]
    arr_time = {t.config.task_id: t.config.arrival_time for t in task_list}
    deadline = {t.config.task_id: t.config.deadline for t in task_list}
    duration = {t.config.task_id: t.config.expected_duration for t in task_list}
    remaining_ids = set(all_ids)

    start_time = {tid: None for tid in all_ids}
    comp_time = {}
    last_tid = None
    preemptions = 0
    context_switches = 0
    busy_time = 0.0
    cpu_idle_time = 0.0

    while remaining_ids:
        action = policy_fn(env, state)
        tid = env.tasks[action].config.task_id

        # Track preemptions and context switches
        if last_tid is not None and tid != last_tid:
            context_switches += 1
            if start_time[tid] is not None:
                preemptions += 1
        last_tid = tid

        # Start time recording
        if start_time[tid] is None:
            start_time[tid] = env.current_time

        before_ids = [t.config.task_id for t in env.tasks]
        prev_time = env.current_time
        state, _, done, _ = env.step(action)
        time_delta = env.current_time - prev_time

        # Track CPU busy time (idle only if no task)
        if action >= len(env.tasks):
            cpu_idle_time += time_delta
        else:
            busy_time += time_delta

        after_ids = [t.config.task_id for t in env.tasks]
        if tid in before_ids and tid not in after_ids:
            comp_time[tid] = env.current_time
            remaining_ids.remove(tid)

        if done:
            break

    N = len(all_ids)
    makespan = env.current_time
    throughput = N / makespan if makespan > 0 else 0
    cpu_utilization = busy_time / makespan if makespan > 0 else 0

    tat = [comp_time.get(tid, env.current_time) - arr_time[tid] for tid in all_ids]
    wait = [start_time.get(tid, env.current_time) - arr_time[tid] for tid in all_ids]
    misses = sum(1 for tid in all_ids if comp_time.get(tid, float('inf')) > deadline[tid])
    avg_burst = sum(duration.values()) / N

    return {
        'avg_turnaround': sum(tat) / N,
        'avg_waiting': sum(wait) / N,
        'miss_rate': misses / N,
        'avg_response': sum(start_time.get(tid, env.current_time) - arr_time[tid] for tid in all_ids) / N,
        'cpu_utilization': cpu_utilization,
    }

# ------ Main Function ------
def main():
    # Load supervised SL model
    sl = TaskPriorityTransformer().to(DEVICE)
    try:
        state_dict_sl = torch.load(
            '/kaggle/input/sgpt/pytorch/default/1/priority_transformer1 (1).pth',
            map_location=DEVICE
        )
        sl.load_state_dict(state_dict_sl)
        print("Loaded SL model successfully.")
    except Exception as e:
        print(f"Could not load SL model: {e}")
    
    # Load RL policy (also used as Titan with different inference strategy)
    rl = PPOPolicy(sl).to(DEVICE)
    try:
        state_dict_rl = torch.load(
            '/kaggle/input/titan-scheduler/pytorch/default/1/rl_scheduler_efficient.pth',
            map_location=DEVICE
        )
        # Fix key mismatch: rename 'pos_encoding' keys to 'positional_encoding'
        fixed_state = {}
        for k, v in state_dict_rl.items():
            new_key = k.replace('pos_encoding', 'positional_encoding')
            fixed_state[new_key] = v
        rl.load_state_dict(fixed_state)
        print("Loaded RL model successfully.")
    except Exception as e:
        print(f"Could not load RL model: {e}")
    
    # Create a results dictionary to store all metrics
    all_results = []
    
    # Run the evaluation
    for ep in range(100):
        seed = 100 + ep
        task_list = generate_task_list(seed)
        
        # Evaluate each policy on the same task list
        m_edf = evaluate_metrics(lambda e, s: edf_heuristic(s), task_list)
        m_sl = evaluate_metrics(lambda e, s: infer_sl(sl, s), task_list)
        m_titan = evaluate_metrics(lambda e, s: infer_titan(rl, s), task_list)
        
        # Store results
        episode_results = {
            'episode': ep + 1,
        }
        for policy_name, metrics in [('edf', m_edf), ('sl', m_sl), ('titan', m_titan)]:
            for key, value in metrics.items():
                episode_results[f'{policy_name}_{key}'] = value
        all_results.append(episode_results)
        
        # Print episode results
        print(f"Episode {ep+1}:")
        print(" EDF:", m_edf)
        print(" SL :", m_sl)
        print(" TITAN:", m_titan)
    
    # Convert results to DataFrame and save to CSV
    results_df = pd.DataFrame(all_results)
    results_df.to_csv('scheduler_evaluation_results.csv', index=False)
    
    # Print average metrics
    print("\nAverage Metrics:")
    avg_metrics = results_df.mean()
    for col in results_df.columns:
        if col != 'episode':
            print(f"{col}: {avg_metrics[col]:.4f}")

if __name__ == '__main__':
    main()

/tmp/ipykernel_31/3523258150.py:266: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict_sl = torch.load(
/tmp/ipykernel_31/3523258150.py:278: FutureWarning: You are us

Loaded SL model successfully.
Loaded RL model successfully.
Episode 1:
 EDF: {'avg_turnaround': 167.98274825391698, 'avg_waiting': 157.90689760219757, 'miss_rate': 0.9375, 'avg_response': 157.90689760219757, 'cpu_utilization': 0.999695085604962}
 SL : {'avg_turnaround': 155.7418828181201, 'avg_waiting': 145.66603216640075, 'miss_rate': 0.9375, 'avg_response': 145.66603216640075, 'cpu_utilization': 0.9653530512601555}
 TITAN: {'avg_turnaround': 194.18622776823474, 'avg_waiting': 44.77831810150061, 'miss_rate': 0.84375, 'avg_response': 44.77831810150061, 'cpu_utilization': 0.9991335915892916}
Episode 2:
 EDF: {'avg_turnaround': 182.694546652649, 'avg_waiting': 171.54886758598738, 'miss_rate': 0.96875, 'avg_response': 171.54886758598738, 'cpu_utilization': 0.9651942650355833}
 SL : {'avg_turnaround': 175.60445354371674, 'avg_waiting': 164.45877447705504, 'miss_rate': 0.9375, 'avg_response': 164.45877447705504, 'cpu_utilization': 0.9654824265014078}
 TITAN: {'avg_turnaround': 221.898708520